# Stage 2 Continued: Fine-Tuning Teeth Detection on DISEASE WITH HEALTHY Data

This notebook continues training the teeth enumeration model on the DISEASE_ONLY dataset prepared in the data preparation notebook. We start from stage 2's last checkpoint, fine-tune it on this narrower dataset, compare checkpoints, then sanity-check predictions the same way as before.

`smart_predict` and `compare_best_vs_last` come from `src/model_utils.py`.

Training here runs live, it's not commented out like in the earlier notebooks. The `on_fit_epoch_end` callback prompts for the current stage number when it's called, worth keeping in mind if this runs somewhere without a console attached, like Kaggle.

In [2]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
# add the project root to the path so the src package can be imported
sys.path.append(os.path.abspath('..'))

import yaml
from ultralytics import YOLO
from src.model_utils import smart_predict,compare_best_vs_last

%matplotlib inline

## Load configs

We'll load the continued stage 2 config, stage 3 config, the data yaml, and the trained models config, then print each one out for a quick look.

In [ ]:
with open('../configs/stage2_continued.yaml','r') as f:

    stage2_cont_config = yaml.safe_load(f)


with open('../configs/stage3.yaml','r') as f:

    stage3_config = yaml.safe_load(f)


# comes from stage2_cont_config, so that has to load first
with open(stage2_cont_config['model_args']['data'],'r') as f:

    enum_cont_data_yaml = yaml.safe_load(f)


with open('../configs/trained_models.yaml','r') as f:

    trained_models_config = yaml.safe_load(f)

In [ ]:
stage2_cont_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/train_quadrant_enumeration.json',
  's2_main_path': '../Data/Processed/Stage 2 (Enumeration Detection)',
  'runs_s2_output': '../Runs/Stage 2'},
 'model_args': {'data': '../Data/Processed/Stage 2 (Enumeration Detection)/data.yaml',
  'model': '../Models/yolo26s.pt',
  'epochs': 70,
  'save': True,
  'imgsz': 1280,
  'batch': 16,
  'patience': 20,
  'optimizer': 'AdamW',
  'lr0': 0.001,
  'lrf': 0.01,
  'plots': True,
  'verbose': True,
  'device': 'cuda',
  'workers': 4,
  'project': 'Runs',
  'name': 'Stage 2',
  'save_dir': '../Runs/Stage 2',
  'auto_augment': 'None',
  'augment': True,
  'mosaic': 0.0,
  'mixup': 0.0,
  'copy_paste': 0.0,
  'cutmix': 0.0,
  'hsv_h': 0.015,
  'hsv_s': 0.4,
  'hsv_v': 0.4,
  'degrees': 5.0,
  'translate': 0.05,
  'scale': 0.1,
  'shear': 0.0,
  'pe

In [ ]:
enum_cont_data_yaml

{'train': '../Data/Processed/Stage 2 Continued (Healthy & Disease Enumeration Detection)/train/images',
 'val': '../Data/Processed/Stage 2 Continued (Healthy & Disease Enumeration Detection)/valid/images',
 'test': '../Data/Processed/Stage 2 Continued (Healthy & Disease Enumeration Detection)/test/images',
 'DISEASE_ONLY': {'train': '../Data/Processed/Stage 2 Continued (Healthy & Disease Enumeration Detection)/Disease Only/train/images',
  'val': '../Data/Processed/Stage 2 Continued (Healthy & Disease Enumeration Detection)/Disease Only/valid/images',
  'test': '../Data/Processed/Stage 2 Continued (Healthy & Disease Enumeration Detection)/Disease Only/test/images'},
 'nc': 8,
 'names': [0, 1, 2, 3, 4, 5, 6, 7]}

In [ ]:
trained_models_config

{'quadrant_detection_model': {'best': '../Runs/Stage 1/weights/best.pt',
  'last': '../Runs/Stage 1/weights/last.pt'},
 'enumeration_detection_model': {'best': '../Runs/Stage 2/weights/best.pt',
  'last': '../Runs/Stage 2/weights/last.pt'},
 'disease_classification_model': {'best': '../Runs/Stage 3/weights/best.pt',
  'last': '../Runs/Stage 3/weights/last.pt'}}

## Fine-tune the model

We load stage 2's last checkpoint as the starting point, then continue training on the DISEASE_ONLY data.

In [ ]:
# start from stage 2's last checkpoint, this is the base for continued training
yolo_model = YOLO(trained_models_config['enumeration_detection_model']['last'])

In [ ]:
# yolo_model.train(**stage2_cont_config['model_args'])

## Compare checkpoints

Now we check best vs last on the test split, passing the original images path and the cleaned annotations dataframe directly this time, along with where to write the results csv.

In [ ]:
# original_images_path and annotations_df_path are passed directly here instead of a results folder
results = compare_best_vs_last(enum_cont_data_yaml,trained_models_config,
                                stage='enum_cont', split_name = 'test',
                                original_images_path=stage2_cont_config['paths']['original_images_path'],
                                annotations_df_path=os.path.join(stage2_cont_config['paths']['s2_cont_main_path'], 'test_df.pkl'),
                                results_csv_path= os.path.join(stage2_cont_config['paths']['runs_s2_cont_output'], 'results.csv'))

Comparison: Best Epoch vs Last Epoch
Best Epoch (42):
  mAP50: 0.9373
  mAP50-95: 0.5620
Last Epoch (62):
  mAP50: 0.9294
  mAP50-95: 0.5610
Verdict (based on training metrics only):
Best Epoch (42) is better than Last Epoch (62).
Difference in mAP50-95: +0.0010
This gap can indicate some overfitting toward the end of training.

Running best and last weights on the test set (stage: enum)
enumeration_detection_model - best


100%|██████████| 248/248 [00:14<00:00, 16.84it/s]


  duplicates: 3, low_confidence: 5, missing: 119, leaks: 0, background: 0, successful: 42, avg_conf: 0.8298
enumeration_detection_model - last


100%|██████████| 248/248 [00:08<00:00, 27.77it/s]

  duplicates: 3, low_confidence: 8, missing: 120, leaks: 0, background: 0, successful: 39, avg_conf: 0.7892

Test set summary
enumeration_detection_model_best: total_errors=127, avg_confidence=0.8298
enumeration_detection_model_last: total_errors=131, avg_confidence=0.7892

Lowest total errors on test set: enumeration_detection_model_best with 127 errors
